# Análise do Tesouro Direto

Notebook que roda o fluxo do projeto seguindo o SEMMA e registra a exploração e a interpretação dos resultados. Funciona no Colab e localmente.

In [ ]:
# No Colab: clona o repositório e instala as dependências. Localmente, pula esse bloco.
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('bi-tesouro-direto'):
    !git clone https://github.com/LeviJesus/bi-tesouro-direto.git
    %cd bi-tesouro-direto
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

## Sample: coleta

In [ ]:
from src import coleta
coleta.coletar()

## Explore: qualidade e estatística descritiva dos dados brutos

In [ ]:
import pandas as pd
bruto = pd.read_csv('data/raw/PrecoTaxaTesouroDireto.csv', sep=';', decimal=',')
print(bruto.shape)
bruto.head()

In [ ]:
print(bruto.isna().sum())
bruto['Tipo Titulo'].value_counts()

In [ ]:
bruto[['Taxa Compra Manha','Taxa Venda Manha','PU Compra Manha','PU Venda Manha']].describe()

## Modify: limpeza e data warehouse

In [ ]:
from src import limpeza, warehouse
tesouro, macro = limpeza.limpar()
warehouse.construir_dw(tesouro, macro)

In [ ]:
con = warehouse.abrir_dw()
con.execute('''
    SELECT d.tipo_titulo, count(distinct d.id_titulo) AS vencimentos, count(*) AS observacoes,
           round(avg(f.taxa_venda), 2) AS taxa_media
    FROM fato_titulo_dia f JOIN dim_titulo d USING (id_titulo)
    GROUP BY 1 ORDER BY 1
''').df()

## Model: KPIs

In [ ]:
from src import metricas
kpis = metricas.calcular_kpis()
metricas.salvar_kpis(kpis)
kpis[['nome_titulo','taxa_atual','rentabilidade_acum','vol_taxa','spread_selic','spread_real_ipca','drawdown_max','duration_anos']].round(2)

## Model e Assess: previsão da taxa do título âncora

In [ ]:
from src import preditivo
prev = preditivo.treinar_e_avaliar()
print(prev['titulo'])
pd.DataFrame(prev['metricas']).round(3)

### Interpretação

Preencher depois de rodar com os dados mais recentes:

- O modelo supera a média móvel de 30 dias? (comparar MAE, RMSE e R² acima)
- O que a projeção de 30 dias úteis sugere para a taxa?
- Limitação: o teste é de previsão um passo à frente, que favorece muito o modelo com defasagem. A projeção recursiva é bem menos confiável.

## Assess: dashboards

In [ ]:
from src import dashboards
dashboards.gerar_dashboard(previsao=prev)